# MobileNetV3-Small Slot Occupancy & Fusion: Large-Scale Training Pipeline

This notebook runs the formal training pipeline on Google Colab (GPU) on a large sample from the official PKLot archive (~4.56 GB), sampled evenly across all 3 lots (`PUCPR`, `UFPR04`, `UFPR05`), weather conditions (`Sunny`, `Cloudy`, `Rainy`), and days.

### Strict Execution Safeguards:
- **Fast-Fail on Error**: Every cell is designed to immediately raise an exception on error, preventing cascade failures.
- **Reproducibility & Traceability**: Clones latest `main`, checks out clean code, and logs `git rev-parse HEAD`.
- **Cache Invalidation**: Automatically deletes stale cached crops (`data/cache/crops`) prior to training/evaluation.
- **Pre-Training Assertion**: Validates that all 3 lots have extracted frames and crops, stopping the run if any lot is missing.
- **Freshness-Gated Drive Backup**: Google Drive backup strictly verifies that checkpoints (`mobilenetv3_loo_fold0.pth`, `fold1`, `fold2`) and `repro_check.json` exist and were produced during the current session.
- **Unified Preprocessing**: Shared `preprocess_slot_crop` across training, evaluation, and inference.


## 0. Training Configuration & Session Initialization
Configure dataset sample size, optional local archive path, and hyperparameters.


In [ ]:
import time
from pathlib import Path

# Record notebook session start timestamp to guard against backing up stale weights
SESSION_START_TIME = time.time()

# Visible top-level configuration
FRAMES_PER_LOT = 150       # Target sampled frames per lot (>=150 frames per lot across Sunny/Cloudy/Rainy/days)
ARCHIVE_PATH = ""          # Optional path to pre-downloaded PKLot archive (e.g., "/content/drive/MyDrive/PKLot.tar.gz"). If empty, downloads official archive.
EPOCHS = 8                 # Epochs per fold
BATCH_SIZE = 64            # Batch size for GPU training
LEARNING_RATE = 3e-4       # Learning rate for AdamW
SEED = 42                  # Global random seed

print("=== Pipeline Configuration ===")
print(f"Frames per lot:  {FRAMES_PER_LOT} (~{FRAMES_PER_LOT * 3} total frames)")
print(f"Archive path:    {ARCHIVE_PATH if ARCHIVE_PATH else '[Download from official PKLot URL]'}")
print(f"Epochs:          {EPOCHS}")
print(f"Batch size:      {BATCH_SIZE}")
print(f"Learning rate:   {LEARNING_RATE}")
print(f"Random seed:     {SEED}")


## 1. Environment Setup & GPU Verification


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("CUDA is not available. Please switch Colab runtime type to GPU (T4 or higher).")


## 2. Clone Latest Repository & Print Git Commit Hash


In [ ]:
import subprocess
import os

repo_dir = "/content/smart-parking-platform"
if os.path.exists(repo_dir):
    subprocess.run(["rm", "-rf", repo_dir], check=True)

subprocess.run(["git", "clone", "https://github.com/bh00mi-11/smart-parking-platform.git", repo_dir], check=True)
os.chdir(os.path.join(repo_dir, "vision"))

# Print current git commit hash for full auditability
commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode("utf-8").strip()
branch_name = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode("utf-8").strip()
print(f"Running on branch: '{branch_name}' at commit: {commit_hash}")

subprocess.run(["pip", "install", "-r", "requirements.txt"], check=True)


## 3. Cache Invalidation & Large Dataset Sampling
Deletes any stale crop caches, downloads or loads the PKLot archive, and extracts a stratified sample evenly across weather and days with stride sampling.


In [ ]:
import shutil
import subprocess

# Preprocessing Safeguard: Delete any cached crops to guarantee fresh preprocessing
cache_dir = Path("data/cache/crops")
if cache_dir.exists():
    shutil.rmtree(cache_dir)
print("Cleared data/cache/crops.")

# Prepare sampling command
cmd = ["python", "scripts/download_data.py", "--frames-per-lot", str(FRAMES_PER_LOT)]
if ARCHIVE_PATH and ARCHIVE_PATH.strip():
    cmd.extend(["--archive-path", ARCHIVE_PATH.strip()])
else:
    cmd.append("--official-archive")

print(f"Executing: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

# Run inspection
subprocess.run(["python", "scripts/inspect_data.py"], check=True)


## 4. Pre-Training Assertion: Dataset Completeness & Frame/Crop Counts
Verifies that every lot has extracted frames and crops, raising an error if any lot is empty.


In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

pklot_dir = Path("data/PKLot")
required_lots = ["PUCPR", "UFPR04", "UFPR05"]

print("=== Pre-Training Dataset Verification ===")
lot_counts = {}

for lot in required_lots:
    lot_dir = pklot_dir / lot
    if not lot_dir.exists():
        raise RuntimeError(f"[DATASET CHECK FAILED] Directory for lot '{lot}' does not exist in {pklot_dir}!")

    jpg_files = list(lot_dir.rglob("*.jpg"))
    xml_files = list(lot_dir.rglob("*.xml"))
    
    if len(jpg_files) == 0:
        raise RuntimeError(f"[DATASET CHECK FAILED] Lot '{lot}' has 0 extracted .jpg frames!")
    if len(jpg_files) != len(xml_files):
        raise RuntimeError(f"[DATASET CHECK FAILED] Lot '{lot}' frame mismatch: {len(jpg_files)} JPGs vs {len(xml_files)} XMLs!")

    # Count total slot annotations
    total_slots = 0
    for xml_p in xml_files:
        tree = ET.parse(xml_p)
        total_slots += len(tree.getroot().findall("space"))

    lot_counts[lot] = {"frames": len(jpg_files), "slots": total_slots}
    print(f"Lot '{lot:<6}': {len(jpg_files):<4} frames | {total_slots:<6} total annotated slots")

print(f"\nAll required lots verified successfully: {sum(v['frames'] for v in lot_counts.values())} total frames, {sum(v['slots'] for v in lot_counts.values())} total slot crops.")


## 5. 3-Fold Leave-One-Lot-Out (LOO) Training
Trains MobileNetV3-small slot classifiers on each fold, saves checkpoints, and exports 50 reference predictions to `repro_check.json`.


In [ ]:
import subprocess

for fold in [0, 1, 2]:
    print(f"\n{'='*70}\nSTARTING LEAVE-ONE-LOT-OUT (LOO) FOLD {fold}\n{'='*70}")
    cmd = [
        "python", "src/train_classifier.py",
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--lr", str(LEARNING_RATE),
        "--protocol", "loo",
        "--fold", str(fold),
        "--seed", str(SEED),
        "--export-repro",
        "--repro-samples", "50"
    ]
    subprocess.run(cmd, check=True)


## 6. Checkpoint Validation & Intermediate Drive Backup (LOO Folds)
Validates that all 3 fold checkpoints and `repro_check.json` exist and were produced during this session, refusing to back up if any are stale.


In [ ]:
import os
import shutil
from pathlib import Path

required_artifacts = [
    "mobilenetv3_loo_fold0.pth",
    "mobilenetv3_loo_fold1.pth",
    "mobilenetv3_loo_fold2.pth",
    "repro_check.json"
]

outputs_dir = Path("outputs")
print("=== Validating LOO Checkpoints & Artifact Freshness ===")
for fname in required_artifacts:
    fpath = outputs_dir / fname
    if not fpath.exists():
        raise FileNotFoundError(f"[BACKUP BLOCKED] Missing required artifact: {fname}")
    mtime = fpath.stat().st_mtime
    if mtime < SESSION_START_TIME:
        raise RuntimeError(f"[BACKUP BLOCKED] Artifact '{fname}' is stale (modified before session start: {mtime} < {SESSION_START_TIME})!")
    print(f"Verified fresh: {fname} (size: {fpath.stat().st_size:,} bytes)")

drive_save_dir = "/content/drive/MyDrive/smart_parking_vision_largerdata_outputs"
os.makedirs(drive_save_dir, exist_ok=True)

for f in outputs_dir.glob("*"):
    if f.is_file() and f.stat().st_mtime >= SESSION_START_TIME:
        dest = os.path.join(drive_save_dir, f.name)
        shutil.copy(str(f), dest)
        print(f"Backed up to Drive: {f.name}")

print(f"\n[SUCCESS] Intermediate backup of fresh LOO fold artifacts completed to {drive_save_dir}!")


## 7. Train Final Production Model (`--protocol all`)
Trains `mobilenetv3_final.pth` across ALL three lots with validation partitioned strictly by day.


In [ ]:
import subprocess

print(f"\n{'='*70}\nSTARTING FINAL MODEL TRAINING (--protocol all)\n{'='*70}")
cmd = [
    "python", "src/train_classifier.py",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--lr", str(LEARNING_RATE),
    "--protocol", "all",
    "--seed", str(SEED)
]
subprocess.run(cmd, check=True)


## 8. Aggregate Results Summary Table


In [ ]:
import json
from pathlib import Path
import pandas as pd

lots = ["PUCPR", "UFPR04", "UFPR05"]
rows = []

for fold, lot in enumerate(lots):
    metrics_file = Path(f"outputs/loo_fold{fold}_{lot}_metrics.json")
    if metrics_file.exists():
        with open(metrics_file) as f:
            m = json.load(f)
        rows.append({
            "Protocol": f"LOO Fold {fold}",
            "Evaluation Lot": lot,
            "Accuracy": f"{m.get('accuracy', 0)*100:.2f}%",
            "Precision": f"{m.get('precision', 0):.4f}",
            "Recall": f"{m.get('recall', 0):.4f}",
            "F1 Score": f"{m.get('f1_score', 0):.4f}",
            "ECE (Uncal)": f"{m.get('ece_uncalibrated', 0):.4f}",
            "ECE (Cal)": f"{m.get('ece_calibrated', 0):.4f}",
            "Latency (ms)": f"{m.get('inference_latency_cpu_ms_per_crop', 0):.2f}"
        })

final_file = Path("outputs/all_lots_final_metrics.json")
if final_file.exists():
    with open(final_file) as f:
        m = json.load(f)
    rows.append({
        "Protocol": "Final Model (All Lots)",
        "Evaluation Lot": "Val Split (PUCPR+UFPR04+UFPR05)",
        "Accuracy": f"{m.get('accuracy', 0)*100:.2f}%",
        "Precision": f"{m.get('precision', 0):.4f}",
        "Recall": f"{m.get('recall', 0):.4f}",
        "F1 Score": f"{m.get('f1_score', 0):.4f}",
        "ECE (Uncal)": f"{m.get('ece_uncalibrated', 0):.4f}",
        "ECE (Cal)": f"{m.get('ece_calibrated', 0):.4f}",
        "Latency (ms)": f"{m.get('inference_latency_cpu_ms_per_crop', 0):.2f}"
    })

df_results = pd.DataFrame(rows)
print("\n=== VISION MODULE EVALUATION SUMMARY ===")
print(df_results.to_markdown(index=False))


## 9. Evaluate YOLO11n Slot Mapping Baseline


In [ ]:
import subprocess
subprocess.run(["python", "src/eval_yolo_lots.py"], check=True)


## 10. Final Backup of All Run Artifacts to Google Drive
Backs up all generated models, metrics, diagrams, and verification files to Google Drive.


In [ ]:
import os
import shutil
from pathlib import Path

outputs_dir = Path("outputs")
drive_save_dir = "/content/drive/MyDrive/smart_parking_vision_largerdata_outputs"
os.makedirs(drive_save_dir, exist_ok=True)

backup_count = 0
for f in outputs_dir.glob("*"):
    if f.is_file() and f.stat().st_mtime >= SESSION_START_TIME:
        dest = os.path.join(drive_save_dir, f.name)
        shutil.copy(str(f), dest)
        print(f"Backed up: {f.name} -> {drive_save_dir}")
        backup_count += 1

print(f"\n[SUCCESS] Final backup completed: {backup_count} fresh artifacts saved to Google Drive!")
